# Midterm Demo · SAP → TFL Shell Generation (GitHub + Colab)

**One Notebook, five visible Agent steps.** The Python code is imported from `tfl_agent`, not pasted here.

**Presenter note:** `MODE = "fixture"` uses a prewritten synthetic LLM response. It validates engineering integration, **not real AI extraction accuracy**. To demonstrate a real LLM call, switch to `MODE = "api"` and configure a Colab Secret (`OPENAI_API_KEY`). Current MVP supports **one Table shell**, not full-study TFL generation.

## 0 · Load Python package from GitHub

**Repository:** `wohaofanaweishenme-cell/tfl-generation-agent`. On a fresh Colab runtime, the next cell clones and installs the package automatically; **no ZIP upload**. (The repository must already contain these files.)

In [ ]:
from pathlib import Path
import os, sys, subprocess

# GitHub repository configured for the midterm demonstration.
REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
PROJECT_ROOT = Path("/content/tfl-generation-agent")

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git clone. Remove or rename it before starting.")

assert (PROJECT_ROOT / "pyproject.toml").exists()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_ROOT}[notebook]"])
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tfl_agent import TFLSession
print("Package loaded:", PROJECT_ROOT)

## 1 · Configuration

Default: safe, reproducible **synthetic fixture** for rehearsals. The optional `api` mode calls an actual LLM and may return errors/review flags; use a real, reviewed Library for meaningful study results. Avoid committing API keys or confidential SAPs to a public repository.

In [ ]:
import json
import pandas as pd
from IPython.display import display, FileLink
from tfl_agent.tools.llm_client import FixtureJSONClient, OpenAIJSONClient

MODE = "fixture"          # "fixture" or "api"
MODEL = "gpt-4.1-mini"    # used only for api mode
TFL_ID = "14.2.1"
TFL_TITLE = "Summary of Overall Survival"
SAP_PATH = PROJECT_ROOT / "input" / "synthetic_sap.pdf"
LIBRARY_PATH = PROJECT_ROOT / "library"
SCHEMA_PATH = PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
OUTPUT_DIR = PROJECT_ROOT / "output" / "midterm_demo"

if MODE == "fixture":
    llm = FixtureJSONClient(PROJECT_ROOT / "examples" / "demo_llm_output.json")
else:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if not key:
        raise RuntimeError("Set OPENAI_API_KEY in Colab Secrets for api mode.")
    os.environ["OPENAI_API_KEY"] = key
    llm = OpenAIJSONClient(model=MODEL)

session = TFLSession(
    sap_path=SAP_PATH, library_path=LIBRARY_PATH,
    schema_path=SCHEMA_PATH, prompts_dir=PROMPTS_DIR,
    output_dir=OUTPUT_DIR, llm=llm, tfl_id=TFL_ID, title=TFL_TITLE,
)
print("Mode:", MODE, "| SAP:", SAP_PATH.name, "| Target:", TFL_TITLE)

## 2 · SAP Parsing
**What the agent sees:** pages, text blocks, selected target TFL, and loaded templates.

In [ ]:
parsed = session.step1_parse()
print(json.dumps(parsed, indent=2, ensure_ascii=False))

## 3 · Evidence-backed Specification Extraction
**Key idea:** find relevant SAP passages and link each extracted requirement to source evidence. For fixture mode, the LLM response is pre-authored.

In [ ]:
spec = session.step2_build_specification()
print("Specification readiness:", spec["readiness"])
print("Schema + evidence validation:", session.spec_validation)
display(pd.DataFrame([spec["statistical_requirements"]]).T.rename(columns={0:"Extracted value"}))
evidence = spec.get("provenance", {}).get("evidence", [])
if evidence:
    display(pd.DataFrame([{
        "Page": e.get("page_numbers"), "SAP section": e.get("section"),
        "Evidence": e.get("exact_text", "")[:180]
    } for e in evidence]).head(8))

## 4 · Template Retrieval
**Key idea:** keyword and metadata matching select a candidate from the Template Library. Scores are retrieval heuristics, not model confidence probabilities.

In [ ]:
retrieval = session.step3_retrieve_templates()
print("Selected template:", retrieval["selected_template_id"])
display(pd.DataFrame(retrieval["candidates"]))
plan = session.preview_modification_plan()
print("Safe to generate:", plan.get("safe_to_generate"))

## 5 · Generate Editable DOCX Table Shell
The generator fills verified placeholders in the selected template without editing the source Library.

In [ ]:
generation = session.step4_generate_shell()
print(json.dumps(generation, indent=2, ensure_ascii=False))

## 6 · Validation & Download
The validator checks schema, source quotation linkage and Word structure. **Passing does not guarantee clinical/statistical correctness.**

In [ ]:
report = session.step5_validate_and_export()
print(json.dumps(report, indent=2, ensure_ascii=False))

if session.shell_path:
    from docx import Document
    shell = Document(session.shell_path)
    print("\nGenerated Word shell preview:")
    print("Title:", shell.paragraphs[0].text if shell.paragraphs else "")
    if shell.tables:
        display(pd.DataFrame([[cell.text for cell in row.cells] for row in shell.tables[0].rows]))
    print("Footnotes:", [p.text for p in shell.paragraphs if "Estimates" in p.text])
    display(FileLink(session.shell_path, result_html_prefix="Download generated DOCX: "))
else:
    print("No DOCX generated; review the validation report and specification issues.")

---
### Demo narrative (≈3 minutes)

1. Explain **input SAP + Template Library**, then point to selected target TFL.
2. Show extracted **analysis population, endpoint, statistics, and SAP evidence**.
3. Show retrieved template ID and why it matches.
4. Open generated DOCX and identify **rows, columns, footnote**.
5. Show validation report and clearly distinguish **synthetic integration test** vs **real model evaluation**.

**Backup:** run the notebook before presenting and keep screenshots and the resulting DOCX locally. This MVP handles one Table; Figure, Listing, and comprehensive automatic target inventory are not yet implemented.